In [27]:
from utils import ByteSeq
import pickle

with open("tokens.pkl", "rb") as f:
    tokens = pickle.load(f)

en_chars = tokens["en_chars"]
en_tokens = tokens["en_tokens"]
yue_chars = tokens["yue_chars"]
yue_tokens = tokens["yue_tokens"]

In [28]:
import torch.nn.functional as F
from torch import nn
import numpy as np
import torch

class Embedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(vocab_size, d_model))

    def forward(self, x):
        return self.weight[x]

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len):
        super().__init__()
        pe = torch.zeros(max_seq_len, d_model)
        for pos in range(max_seq_len):
            for k in range(d_model // 2):
                freq = pos / (10000 ** (2*k / d_model))
                pe[pos, 2*k] = np.sin(freq)
                pe[pos, 2*k+1] = np.cos(freq)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        return x + self.pe[:x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, seq_len, n_heads, masked = False):
        super().__init__()
        self.d_k = d_model // n_heads
        self.n_heads = n_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model) 
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.register_buffer('mask', torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool())
        self.masked = masked

    def forward(self, x, cross=None):
        B, S, D = x.shape
        cross = cross if cross is not None else x
        Q = self.W_Q(x)
        K = self.W_K(cross)
        V = self.W_V(cross)
        
        # want B, n_heads, S, d_k
        Q = Q.view(B, S, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(B, cross.size(1), self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(B, cross.size(1), self.n_heads, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / (self.d_k ** 0.5)
        if self.masked:
            scores = scores.masked_fill(self.mask[:S, :S], float('-inf'))
        out = F.softmax(scores, dim=-1) @ V
        # back to B, S, D
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        return self.W_O(out)

class Encoder(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, d_ff = 2048):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.source_MHA = MultiHeadAttention(d_model, max_seq_len, n_heads, masked=False)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        x = x + self.source_MHA(self.norm1(x))
        x = x + self.ff(self.norm2(x))
        return x

class Decoder(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, d_ff = 2048):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.masked_MHA = MultiHeadAttention(d_model, max_seq_len, n_heads, masked=True)
        self.cross_MHA = MultiHeadAttention(d_model, max_seq_len, n_heads, masked=False)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, target, cross):
        target = target + self.masked_MHA(self.norm1(target))
        target = target + self.cross_MHA(self.norm2(target), cross)
        target = target + self.ff(self.norm3(target))
        return target


class Transformer(nn.Module):
    def __init__(self, source_vocab_size, target_vocab_size, d_model, n_heads, source_l_layers, target_l_layers, max_seq_len):
        super().__init__()
        self.source_embed = Embedding(source_vocab_size, d_model)
        self.target_embed = Embedding(target_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_seq_len)
        self.encoder_layers = nn.ModuleList([Encoder(d_model, n_heads, max_seq_len) for _ in range(source_l_layers)])
        self.decoder_layers = nn.ModuleList([Decoder(d_model, n_heads, max_seq_len) for _ in range(target_l_layers)])
        self.output_proj = nn.Linear(d_model, target_vocab_size)
    
    def encoder(self, source):
        source = self.pos_enc(self.source_embed(source))
        for layer in self.encoder_layers:
            source = layer(source)
        return source

    def decoder(self, target, enc_out):
        target = self.pos_enc(self.target_embed(target))
        for layer in self.decoder_layers:
            target = layer(target, enc_out)
        logits = self.output_proj(target)
        if self.training:
            return logits
        return F.softmax(logits, dim=-1)

    def forward(self, source, target):
        enc_out = self.encoder(source)
        dec_out = self.decoder(target, enc_out)
        return dec_out

In [29]:
from utils import token_split

PAD_TOKEN = "<PAD>"
START_TOKEN = "<START>"
END_TOKEN = "<END>"
PAD_ID = 0

en_vocab_list = [PAD_TOKEN] + sorted(en_chars) + en_tokens + [START_TOKEN, END_TOKEN]
yue_vocab_list = [PAD_TOKEN] + sorted(yue_chars) + yue_tokens + [START_TOKEN, END_TOKEN]

en_vocab_list = sorted(en_chars) + en_tokens + [START_TOKEN, END_TOKEN]
en_vocab = {token: i for i, token in enumerate(en_vocab_list)}

yue_vocab_list = sorted(yue_chars) + yue_tokens + [START_TOKEN, END_TOKEN]
yue_vocab = {token: i for i, token in enumerate(yue_vocab_list)}


def encode(text, merges, vocab):
    merge_priority = {merge: i for i, merge in enumerate(merges)}
    words = token_split(text)
    all_ids = []
    for word in words:
        tokens = [ByteSeq(b) for b in word.encode("utf-8")]
        while len(tokens) > 1:
            best = None
            best_idx = None
            for i in range(len(tokens) - 1):
                pair = tokens[i] + tokens[i+1]
                if pair in merge_priority:
                    if best is None or merge_priority[pair] < merge_priority[best]:
                        best = pair
                        best_idx = i
            if best is None:
                break
            tokens[best_idx] = best
            tokens.pop(best_idx + 1)
        for t in tokens:
            all_ids.append(vocab[t])
    return all_ids

In [30]:
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

class TranslationDataset(Dataset):
    def __init__(self, en_file, yue_file, en_merges, en_vocab, yue_merges, yue_vocab):
        with open(en_file) as f:
            en_lines = f.read().strip().split("\n")
        with open(yue_file) as f:
            yue_lines = f.read().strip().split("\n")

        self.data = []
        for en, yue in tqdm(zip(en_lines, yue_lines), total = len(en_lines)):
            src = torch.tensor(encode(en, en_merges, en_vocab))
            tgt = torch.tensor(
                [yue_vocab[START_TOKEN]] +
                encode(yue, yue_merges, yue_vocab) +
                [yue_vocab[END_TOKEN]]
            )
            self.data.append((src, tgt))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    srcs, tgts = zip(*batch)
    srcs = nn.utils.rnn.pad_sequence(srcs, batch_first=True, padding_value=PAD_ID)
    tgts = nn.utils.rnn.pad_sequence(tgts, batch_first=True, padding_value=PAD_ID)
    return srcs, tgts


dataset = TranslationDataset("en.txt", "yue.txt", en_tokens, en_vocab, yue_tokens, yue_vocab)

train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_set, batch_size=32, collate_fn=collate_fn)
test_loader = DataLoader(test_set, batch_size=32, collate_fn=collate_fn)

100%|██████████| 41859/41859 [07:28<00:00, 93.31it/s] 


In [ ]:
model = Transformer(
    source_vocab_size=len(en_vocab),
    target_vocab_size=len(yue_vocab),
    d_model=512,
    n_heads=8,
    source_l_layers=6,
    target_l_layers=6,
    max_seq_len=512
)

device = torch.device("mps")
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)

EPOCHS = 20
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for src, tgt in tqdm(train_loader):
        src, tgt = src.to(device), tgt.to(device)
        tgt_input = tgt[:, :-1]
        tgt_label = tgt[:, 1:]

        logits = model(src, tgt_input)
        loss = loss_fn(logits.reshape(-1, len(yue_vocab)), tgt_label.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for src, tgt in val_loader:
            src, tgt = src.to(device), tgt.to(device)
            tgt_input = tgt[:, :-1]
            tgt_label = tgt[:, 1:]
            logits = model(src, tgt_input)
            val_loss += loss_fn(logits.reshape(-1, len(yue_vocab)), tgt_label.reshape(-1)).item()

    print(f"Epoch {epoch+1} | Train loss: {total_loss/len(train_loader):.4f} | Val loss: {val_loss/len(val_loader):.4f}")

  2%|▏         | 19/1047 [02:04<1:52:21,  6.56s/it]


KeyboardInterrupt: 